# ETL Pipeline Demo — Bibliometrix Python
## Advanced Level 

This notebook demonstrates the ETL pipeline developed for the Bibliometrix-Python project.
The pipeline extracts data from OpenAlex and PubMed APIs, transforms it into the WoS standard schema, and validates the output.

---
## PHASE 1: EXTRACT
Data is retrieved via REST APIs from OpenAlex and PubMed.
The `retrieve()` function handles pagination, rate limits, and retries automatically.

In [2]:
from www.services.api_retriever import retrieve

print("=== EXTRACT: OpenAlex ===")
records_oa = retrieve(query="machine learning", platform="openalex", total=10)
print(f"Records retrieved: {len(records_oa)}")
print(f"Sample raw keys: {list(records_oa[0].keys())[:8]}")
print(f"\nSample title: {records_oa[0].get('title', 'N/A')}")

=== EXTRACT: OpenAlex ===
Records retrieved: 10
Sample raw keys: ['id', 'doi', 'title', 'display_name', 'relevance_score', 'publication_year', 'publication_date', 'ids']

Sample title: Scikit-learn: Machine Learning in Python


In [3]:
print("=== EXTRACT: PubMed ===")
records_pm = retrieve(query="machine learning", platform="pubmed", total=10)
print(f"Records retrieved: {len(records_pm)}")
print(f"Sample raw keys: {list(records_pm[0].keys())[:8]}")
print(f"\nSample title: {records_pm[0].get('Title', 'N/A')}")

=== EXTRACT: PubMed ===
Records retrieved: 10
Sample raw keys: ['uid', 'pubdate', 'epubdate', 'source', 'authors', 'lastauthor', 'title', 'sorttitle']

Sample title: N/A


---
## PHASE 2: TRANSFORM
Raw API responses are mapped to the WoS standard schema using mapping dictionaries.
Multi-value fields are cast to `list[str]`, scalar fields to `str`, and `TC` to `int`.

In [4]:
from www.services.standardizer import standardize
import pandas as pd

print("=== TRANSFORM: OpenAlex ===")
df_oa = standardize(records_oa, source="openalex")
print(f"Shape: {df_oa.shape}")
print(f"Columns: {df_oa.columns.tolist()}")
df_oa[['AU', 'TI', 'PY', 'SO', 'TC', 'DB']].head(3)

=== TRANSFORM: OpenAlex ===
Shape: (10, 26)
Columns: ['UT', 'DI', 'TI', 'PY', 'LA', 'DT', 'TC', 'SO', 'JI', 'AU', 'AF', 'C1', 'RP', 'AB', 'VL', 'IS', 'BP', 'EP', 'DE', 'AU_CO', 'CR', 'ID', 'PMID', 'DB', 'SR', 'SR_FULL']


,AU,TI,PY,SO,TC,DB
0,"[Fabián Pedregosa, Gaël Varoquaux, Alexandre G...",Scikit-learn: Machine Learning in Python,2012,ARXIV (CORNELL UNIVERSITY),63730,OPENALEX
1,[],"Genetic algorithms in search, optimization, an...",1989,CHOICE REVIEWS ONLINE,49334,OPENALEX
2,[J. R. Quinlan],C4.5: Programs for Machine Learning,1992,,23698,OPENALEX


In [5]:
print("=== TRANSFORM: PubMed ===")
df_pm = standardize(records_pm, source="pubmed")
print(f"Shape: {df_pm.shape}")
print(f"Columns: {df_pm.columns.tolist()}")
df_pm[['AU', 'TI', 'PY', 'SO', 'TC', 'DB']].head(3)

=== TRANSFORM: PubMed ===
Shape: (10, 26)
Columns: ['UT', 'TI', 'SO', 'JI', 'PY', 'VL', 'IS', 'LA', 'DT', 'RP', 'AU', 'AF', 'DI', 'PMID', 'BP', 'EP', 'CR', 'AB', 'C1', 'AU_CO', 'DE', 'ID', 'TC', 'DB', 'SR', 'SR_FULL']


,AU,TI,PY,SO,TC,DB
0,"[Zhang L, Bi S, Liu X, Sun Q, Lu X, Cheng H]",Carbonyl-Modulated Lowest Unoccupied Molecular...,2026,"Advanced materials (Deerfield Beach, Fla.)",0,PUBMED
1,"[Ma C, Zhao X, Li J, Liu X, Zhang M, Liu Z, Wu...",Interfacial Regulation-Driven Dual-Enrichment ...,2026,"Small (Weinheim an der Bergstrasse, Germany)",0,PUBMED
2,"[Niu M, Song J, Liu G]",Nanoscale Tailoring of Bulk High Entropy Alloy...,2026,"Small (Weinheim an der Bergstrasse, Germany)",0,PUBMED


### Inspect multi-value fields
Author keywords (`DE`) and cited references (`CR`) must be `list[str]`.

In [6]:
print("=== Multi-value fields (OpenAlex) ===")
print(f"AU type: {type(df_oa['AU'].iloc[0])}")
print(f"AU sample: {df_oa['AU'].iloc[0]}")
print(f"\nDE type: {type(df_oa['DE'].iloc[0])}")
print(f"DE sample: {df_oa['DE'].iloc[0]}")
print(f"\nCR type: {type(df_oa['CR'].iloc[0])}")
print(f"CR sample (first 2): {df_oa['CR'].iloc[0][:2]}")

=== Multi-value fields (OpenAlex) ===
AU type: <class 'list'>
AU sample: ['Fabián Pedregosa', 'Gaël Varoquaux', 'Alexandre Gramfort', 'Vincent Michel', 'Bertrand Thirion', 'Olivier Grisel', 'Mathieu Blondel', 'Müller, Andreas', 'Nothman, Joel', 'Louppe, Gilles', 'Peter Prettenhofer', 'Ron J. Weiss', 'Vincent Dubourg', 'Jake Vanderplas', 'Alexandre Passos', 'David Cournapeau', 'Matthieu Brucher', 'Matthieu Perrot', 'Édouard Duchesnay']

DE type: <class 'list'>
DE sample: ['Python (programming language)', 'Documentation', 'Computer science', 'MIT License', 'Artificial intelligence', 'Machine learning', 'Programming language', 'License', 'Software engineering', 'Operating system']

CR type: <class 'list'>
CR sample (first 2): ['Chang C, 2011, ACM TRANSACTIONS ON INTELLIGENT SYSTEMS AND TECHNOLOGY', 'Friedman J, 2010, PUBMED']


---
## PHASE 3: VALIDATE
The validation module checks:
1. All mandatory columns exist
2. No NaN or None values remain
3. Multi-value columns are correctly typed as lists

In [7]:
from www.services.validator import validate

print("=== VALIDATE: OpenAlex ===")
df_oa = validate(df_oa)
print(f"\nSR sample: {df_oa['SR'].iloc[0]}")
df_oa[['SR', 'DE', 'AB']].head(3)

=== VALIDATE: OpenAlex ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.

SR sample: Pedregosa F, 2012, ARXIV (CORNELL UNIVERSITY)


,SR,DE,AB
0,"Pedregosa F, 2012, ARXIV (CORNELL UNIVERSITY)","[Python (programming language), Documentation,...",Scikit-learn is a Python module integrating a ...
1,"NA, 1989, CHOICE REVIEWS ONLINE","[Computer science, Artificial intelligence, Ma...",From the Publisher:\r\nThis book brings togeth...
2,"Quinlan J, 1992,","[Computer science, Unix, Classifier (UML), Mac...",Classifier systems play a major role in machin...


In [8]:
print("=== VALIDATE: PubMed ===")
df_pm = validate(df_pm)
print(f"\nSR sample: {df_pm['SR'].iloc[0]}")
df_pm[['SR', 'DE', 'AB']].head(3)

=== VALIDATE: PubMed ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.

SR sample: L Z, 2026, ADV MATER


,SR,DE,AB
0,"L Z, 2026, ADV MATER",[],
1,"C M, 2026, SMALL",[],
2,"M N, 2026, SMALL",[],


---
## FULL PIPELINE — 200 records
End-to-end demonstration with 200 records per platform, exported to CSV.

In [13]:
print("=== FULL PIPELINE: OpenAlex (200 records) ===")
records_oa_200 = retrieve(query="machine learning", platform="openalex", total=200)
df_oa_200 = standardize(records_oa_200, source="openalex")
df_oa_200 = validate(df_oa_200)
df_oa_200.to_csv("test_openalex_200.csv", index=False)
print(f"Shape: {df_oa_200.shape}")
print("CSV saved: test_openalex_200.csv")
df_oa_200[['AU', 'TI', 'PY', 'SO', 'TC', 'SR']].head(10)

=== FULL PIPELINE: OpenAlex (200 records) ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.
Shape: (200, 26)
CSV saved: test_openalex_200.csv


,AU,TI,PY,SO,TC,SR
0,"[Fabián Pedregosa, Gaël Varoquaux, Alexandre G...",Scikit-learn: Machine Learning in Python,2012,ARXIV (CORNELL UNIVERSITY),63730,"Pedregosa F, 2012, ARXIV (CORNELL UNIVERSITY)"
1,[],"Genetic algorithms in search, optimization, an...",1989,CHOICE REVIEWS ONLINE,49334,"NA, 1989, CHOICE REVIEWS ONLINE"
2,[J. R. Quinlan],C4.5: Programs for Machine Learning,1992,,23698,"Quinlan J, 1992,"
3,[Arthur Asuncion],UCI Machine Learning Repository,2007,MEDICAL ENTOMOLOGY AND ZOOLOGY,24350,"Asuncion A, 2007, MEDICAL ENTOMOLOGY AND ZOOLOGY"
4,"[Ian H. Witten, Eibe Frank, Mark A. Hall]",Data Mining: Practical Machine Learning Tools ...,2011,ELSEVIER EBOOKS,25713,"Witten I, 2011, ELSEVIER EBOOKS"
5,[Nasser M. Nasrabadi],Pattern Recognition and Machine Learning,2007,JOURNAL OF ELECTRONIC IMAGING,22083,"Nasrabadi N, 2007, JOURNAL OF ELECTRONIC IMAGING"
6,[David E. Goldberg],"Genetic Algorithms in Search, Optimization and...",1988,,17771,"Goldberg D, 1988,"
7,[],Proceedings of the 24th international conferen...,2007,,11734,"NA, 2007,"
8,"[Carl Edward Rasmussen, Christopher K. I. Will...",Gaussian Processes for Machine Learning,2005,THE MIT PRESS EBOOKS,10489,"Rasmussen C, 2005, THE MIT PRESS EBOOKS"
9,[Kevin P. Murphy],Machine learning a probabilistic perspective,2012,,9328,"Murphy K, 2012,"


In [11]:
print("=== FULL PIPELINE: PubMed (200 records) ===")
records_pm_200 = retrieve(query="machine learning", platform="pubmed", total=200, mindate="2015", maxdate="2024")
df_pm_200 = standardize(records_pm_200, source="pubmed")
df_pm_200 = validate(df_pm_200)
df_pm_200.to_csv("test_pubmed_200.csv", index=False)
print(f"Shape: {df_pm_200.shape}")
print("CSV saved: test_pubmed_200.csv")
df_pm_200[['AU', 'TI', 'PY', 'SO', 'TC', 'SR']].head(10)

=== FULL PIPELINE: PubMed (200 records) ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.
Shape: (200, 26)
CSV saved: test_pubmed_200.csv


,AU,TI,PY,SO,TC,SR
0,"[Figueroa-Quiñones J, Ipanaque-Neyra J, Gómez ...","Development, validation and use of artificial-...",2023,F1000Research,0,"J F, 2023, F1000RES"
1,"[de Mattos BP, Mattjie C, Ravazio R, Barros RC...",Craving for a Robust Methodology: A Systematic...,2026,International journal of mental health and add...,0,"BP d, 2026, INT J MENT HEALTH ADDICT"
2,"[Kuang A, Yu Y, Siddique J, Scholtens D]",Imputation of Missing Continuous Glucose Monit...,2026,Journal of diabetes science and technology,0,"A K, 2026, J DIABETES SCI TECHNOL"
3,"[Marsico C, Renteria C, Grimm JR, Fernandez-Ar...",A Machine Learning Approach to Quantitative An...,2025,Small structures,0,"C M, 2025, SMALL STRUCT"
4,"[Upadhyay K, Fuhg JN, Bouklas N, Ramesh KT]",Physics-informed data-driven discovery of cons...,2026,Computational mechanics,0,"K U, 2026, COMPUT MECH"
5,"[Zhan Z, Zhou S, Deng J, Zhang R]",Improving electronic health record processing ...,2024,AMIA ... Annual Symposium proceedings. AMIA Sy...,0,"Z Z, 2024, AMIA ANNU SYMP PROC"
6,"[Xie Y, Cui H, Zhang Z, Lu J, Shu K, Nahab F, ...",KERAP: A Knowledge-Enhanced Reasoning Approach...,2024,AMIA ... Annual Symposium proceedings. AMIA Sy...,0,"Y X, 2024, AMIA ANNU SYMP PROC"
7,"[Sivarajkumar S, Ameri K, Li C, Wang Y, Jiang M]",Automating Adjudication of Cardiovascular Even...,2024,AMIA ... Annual Symposium proceedings. AMIA Sy...,0,"S S, 2024, AMIA ANNU SYMP PROC"
8,"[Wang M, Kuan YH, Alba PR, Gan Q, Schoen MW, T...",Developing Large Language Model-based Pipeline...,2024,AMIA ... Annual Symposium proceedings. AMIA Sy...,0,"M W, 2024, AMIA ANNU SYMP PROC"
9,"[Nguyen QN, Wu H, Pontikos N, Wang SY]",Addressing Generalizability in Clinical Named ...,2024,AMIA ... Annual Symposium proceedings. AMIA Sy...,0,"QN N, 2024, AMIA ANNU SYMP PROC"


In [14]:
df_oa_200.to_excel("test_openalex_200.xlsx", index=False) 

df_pm_200.to_excel("test_pubmed_200.xlsx", index=False)  


---
## Summary

| Platform | Records | Columns | NaN | SR |
|----------|---------|---------|-----|----|
| OpenAlex | 200 | 26 | 0 | ✅ |
| PubMed   | 200 | 26 | 0 | ✅ |

The ETL pipeline successfully:
- Extracted data from OpenAlex and PubMed REST APIs
- Transformed raw JSON into the WoS standard schema
- Enforced type contracts (list[str], str, int)
- Validated all mandatory columns
- Generated standardized CSV files ready for Bibliometrix-Python analysis

In [12]:
print(df_pm_200['PY'].value_counts().sort_index())

PY
2023      4
2024    139
2025     53
2026      4
Name: count, dtype: int64
